# UVERA · NB06 — AI-6 Case Linker

| | |
|---|---|
| **Accelerator** | None (CPU) |
| **Internet** | On |
| **Secret** | — |
| **Inputs to add** | — (uses AI-1 and AI-5 results from the repo; run **after** NB01 and NB05 results are committed) |
| **Expected runtime** | ~5–15 min |
| **How to run** | **Save Version → Save & Run All (Commit)** — runs in the background, no 20-minute idle timeout, 12-hour cap |

**What this AI does (say this to judges):** Instead of five separate alerts about one scam, AI-6 follows the money forward in time (up to 3 hops, 48 hours) from each risky transfer to where it was cashed out (agents) or spent (QR merchants), and joins related alerts into one case with an evidence graph. The graph itself is the explanation the analyst sees.

**Outputs** (in `/kaggle/working/outputs/NB06_outputs.zip`): `artifacts/ai6/cases.json`, `reports/metrics_ai6.json`

All data is synthetic or open-licensed. No real customer data.

In [ ]:
# 1) Get the UVERA code from the public GitHub repo (nothing to upload by hand)
REPO = "https://github.com/ABRUBAB/takaresolve-ai-hackathon-2026.git"
REF = "main"   # replace with a commit hash to reproduce an exact run
import os, sys, subprocess
if not os.path.exists("/tmp/uvera"):
    subprocess.run(["git", "clone", "-q", REPO, "/tmp/uvera"], check=True)
subprocess.run(["git", "-C", "/tmp/uvera", "fetch", "-q", "origin", REF], check=False)
subprocess.run(["git", "-C", "/tmp/uvera", "checkout", "-q", REF], check=True)
COMMIT = subprocess.run(["git", "-C", "/tmp/uvera", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
os.environ["UVERA_ROOT"] = "/tmp/uvera"
sys.path.insert(0, "/tmp/uvera/ml")   # use Kaggle's own numpy/pandas/sklearn/lightgbm (no reinstall)
print("UVERA code at commit", COMMIT)
print(f'To reproduce this exact run later, set REF = "{COMMIT}" in this cell.')

In [ ]:
# 2) Reproducibility: seeds, versions, time budget, output folder
import json
from IPython.display import Image, display
from uvera_ml.common import env_info, set_seed, Budget, out_dir, zip_outputs, write_json
set_seed(42)
budget = Budget(minutes=60)
OUT = out_dir("NB06")
info = env_info(); info["uvera_commit"] = COMMIT
write_json(OUT / "reports" / "versions_nb06.json", info)
print(json.dumps(info, indent=1))

In [ ]:
# 3) Build the synthetic world. Same seed -> identical data on every Kaggle account (checked by hash).
from pathlib import Path
from uvera_ml.sim.world import load_or_generate
world = load_or_generate("/tmp/world_full", scale="full")
print(json.dumps(world.meta, indent=1)[:1500])
ref = Path("/tmp/uvera/reports/world_meta.json")
if ref.exists():
    expected = json.loads(ref.read_text(encoding="utf-8"))["meta"]["hashes"]["events"]
    print("Data identical to NB00 run:", expected == world.meta["hashes"]["events"])
budget.check("world")

## Load AI-1 and AI-5 results
From the repo if they are committed; otherwise a quick re-run is done here (and noted).

In [ ]:
import pandas as pd
from pathlib import Path
a1 = Path("/tmp/uvera/artifacts/ai1/test_scores.parquet")
a5 = Path("/tmp/uvera/artifacts/ai5/merchant_scores.parquet")
if a1.exists():
    scored = pd.read_parquet(a1); print("AI-1 scores from repo")
else:
    from uvera_ml.models.ai1 import run_ai1
    run_ai1(world, "/tmp/ai1_quick", quick=True)
    scored = pd.read_parquet("/tmp/ai1_quick/artifacts/ai1/test_scores.parquet"); print("AI-1 quick re-run (repo results missing)")
if a5.exists():
    merchants = pd.read_parquet(a5); print("AI-5 scores from repo")
else:
    from uvera_ml.models.ai5 import run_ai5
    run_ai5(world, "/tmp/ai5_quick", quick=True)
    merchants = pd.read_parquet("/tmp/ai5_quick/artifacts/ai5/merchant_scores.parquet"); print("AI-5 quick re-run (repo results missing)")

In [ ]:
from uvera_ml.graph.linker import run_ai6
summary = run_ai6(world, OUT, scored, merchants)
print(json.dumps(summary, indent=1, default=float))
cases = json.loads((OUT / "artifacts" / "ai6" / "cases.json").read_text(encoding="utf-8"))
c = cases[0]
print(c["case_key"], "score", c["score"], "| victims", len(c["victims"]), "| wallets", c["wallets"], "| merchants", c["merchants"], "| agents", c["agents"])
for e in c["evidence"]:
    print(" -", e["id"], e["type"], e["text"])
budget.check("ai6")

In [ ]:
# Last step: package the results. Download NB06_outputs.zip from the Output tab.
budget.check("done")
zip_outputs(OUT)